In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# QA answer-level semantic entropy (BioASQ Task B + SQuAD 2.0)

**Measurement:** answer-level semantic entropy via bidirectional NLI clustering of generated answers.
This is the original semantic-entropy formulation — **not** the MedMentions/CADEC CUI-entropy grid.

**Hard constraints**
- NO SapBERT / UMLS / CUI mapping
- Generatives only: BioMistral-7B, Mistral-7B-Instruct-v0.1, OpenBioLLM-8B, Meta-Llama-3-8B-Instruct, FLAN-T5-base
- Perturb the **question** through gates G1–G5; **G6 OFF**
- Save mean per-token logprob of the original answer as confidence
- Deterministic greedy decode, `max_new_tokens=32`, chat templates for causal models
- Inclusion: `m >= 3` accepted perturbations (≥4 total outputs)

Kernel: `torch_gpu` (`~/.conda/envs/torch_gpu/bin/python`)


In [1]:
# =============================================================================
# 1) CFG
# =============================================================================
from pathlib import Path
import os

PROJECT_ROOT = PROJECT_ROOT

CFG = {
    # Local generative model dirs / HF ids (expanded below)
    "models": {
        "biomistral": str(Path.home() / "data/models/BioMistral-7B"),
        "mistral": str(Path.home() / "data/models/Mistral-7B-Instruct-v0.1"),
        "openbiollm": str(Path.home() / "data/models/Llama3-OpenBioLLM-8B"),
        "llama3": str(Path.home() / "data/models/Meta-Llama-3-8B-Instruct"),
        "flan-t5-base": "google/flan-t5-base",
    },
    # Datasets (local archives — Task B training + SQuAD2 validation)
    "squad_path": str(PROJECT_ROOT / "Datasets" / "SQuAD2.0"),          # zip: validation.csv
    "bioasq_path": str(PROJECT_ROOT / "Datasets" / "BioASQ-training13b.zip"),  # training13b.json
    # NLI (reuse G2 entailment model)
    "nli_model": "cross-encoder/nli-MiniLM2-L6-H768",
    "out_dir": str(PROJECT_ROOT / "outputs" / "qa"),
    "nli_threshold": 0.72,
    "min_m": 3,
    "squad_subsample": 1000,
    "seed": 13,
    "max_new_tokens": 32,
    "k_perturb_attempts": 8,   # generator attempts per question (G1–G5 filter)
}

# Expand ~
for k, v in list(CFG["models"].items()):
    CFG["models"][k] = os.path.expanduser(v)
CFG["squad_path"] = os.path.expanduser(CFG["squad_path"])
CFG["bioasq_path"] = os.path.expanduser(CFG["bioasq_path"])
CFG["out_dir"] = os.path.expanduser(CFG["out_dir"])

OUT_DIR = Path(CFG["out_dir"])
# Smoke runs MUST use a separate namespace so they never touch full-run caches
import os as _os_cfg
_SMOKE = _os_cfg.environ.get("QA_SMOKE", "").strip() in {"1", "true", "True"}
INTER_DIR = OUT_DIR / ("intermediate_smoke" if _SMOKE else "intermediate")
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTER_DIR.mkdir(parents=True, exist_ok=True)
if _SMOKE:
    print(f"QA_SMOKE cache namespace: {INTER_DIR}")

MODEL_SPECS = [
    {"key": "biomistral", "name": "BioMistral-7B", "arch": "causal"},
    {"key": "mistral", "name": "Mistral-7B-Instruct-v0.1", "arch": "causal"},
    {"key": "openbiollm", "name": "OpenBioLLM-8B", "arch": "causal"},
    {"key": "llama3", "name": "Meta-Llama-3-8B-Instruct", "arch": "causal"},
    {"key": "flan-t5-base", "name": "FLAN-T5-base", "arch": "seq2seq"},
]

print("CFG ready")
print(f"  out_dir={OUT_DIR}")
print(f"  nli_threshold={CFG['nli_threshold']}  min_m={CFG['min_m']}  squad_subsample={CFG['squad_subsample']}")
for s in MODEL_SPECS:
    print(f"  {s['name']:28s}  key={s['key']}  src={CFG['models'][s['key']]}")


CFG ready
  out_dir=/home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa
  nli_threshold=0.72  min_m=3  squad_subsample=1000
  BioMistral-7B                 key=biomistral  src=/home/s224858267/data/models/BioMistral-7B
  Mistral-7B-Instruct-v0.1      key=mistral  src=/home/s224858267/data/models/Mistral-7B-Instruct-v0.1
  OpenBioLLM-8B                 key=openbiollm  src=/home/s224858267/data/models/Llama3-OpenBioLLM-8B
  Meta-Llama-3-8B-Instruct      key=llama3  src=/home/s224858267/data/models/Meta-Llama-3-8B-Instruct
  FLAN-T5-base                  key=flan-t5-base  src=google/flan-t5-base


In [2]:
# =============================================================================
# Setup: imports + load_generative (reuse CADEC / MedMentions pattern)
# =============================================================================
import ast
import gc
import json
import math
import re
import string
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    MarianMTModel,
    MarianTokenizer,
    T5ForConditionalGeneration,
    T5Tokenizer,
    pipeline,
)

assert torch.cuda.is_available(), "CUDA required"

DEVICE = "cuda"


def _log(msg: str) -> None:
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"[{ts}] {msg}", flush=True)


def assert_model_on_cuda(model, name: str) -> None:
    devices = {p.device for p in model.parameters()}
    cuda_devs = {d for d in devices if d.type == "cuda"}
    assert cuda_devs, f"CPU PLACEMENT BUG: {name} devices={devices}"
    print(f"DEVICE OK [{name}]: {sorted(str(d) for d in devices)} | GPU={torch.cuda.get_device_name(0)}")


def load_generative(name_or_key: str):
    """Return (model, tok). Accepts MODEL_SPECS key or display name."""
    key = name_or_key
    arch = "causal"
    for s in MODEL_SPECS:
        if name_or_key in (s["key"], s["name"]):
            key, arch = s["key"], s["arch"]
            break
    src = CFG["models"][key]
    if arch == "seq2seq":
        tok = AutoTokenizer.from_pretrained(src)
        model = AutoModelForSeq2SeqLM.from_pretrained(src, torch_dtype=torch.bfloat16)
        model = model.to("cuda").eval()
        assert_model_on_cuda(model, f"seq2seq:{key}")
        return model, tok
    tok = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        src, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
    )
    assert_model_on_cuda(model, f"causal:{key}")
    return model, tok


def free_model(model) -> None:
    del model
    gc.collect()
    torch.cuda.empty_cache()


print(f"GPU: {torch.cuda.get_device_name(0)}")


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA L40S


In [3]:
# =============================================================================
# 2) Dataset adapters -> uniform record
#    {id, question, context, gold_answers:list[str], is_unanswerable:bool, qtype}
# =============================================================================

def _flatten_answers(obj) -> list[str]:
    """Flatten nested BioASQ exact_answer lists to unique non-empty strings."""
    out: list[str] = []
    if obj is None:
        return out
    if isinstance(obj, str):
        s = obj.strip()
        return [s] if s else []
    if isinstance(obj, (list, tuple)):
        for x in obj:
            out.extend(_flatten_answers(x))
        # preserve order, drop dups
        seen, uniq = set(), []
        for a in out:
            if a and a not in seen:
                seen.add(a)
                uniq.append(a)
        return uniq
    s = str(obj).strip()
    return [s] if s else []


def _parse_squad_answers_cell(raw) -> list[str]:
    """Parse HF-export CSV answers cell (may embed numpy-ish text)."""
    if raw is None or (isinstance(raw, float) and math.isnan(raw)):
        return []
    if isinstance(raw, dict):
        texts = raw.get("text", [])
        if hasattr(texts, "tolist"):
            texts = texts.tolist()
        return [str(t).strip() for t in texts if str(t).strip()]
    s = str(raw)
    # Prefer datasets load; CSV fallback: extract quoted spans after 'text'
    try:
        # common: "{'text': array([...], dtype=object), ...}"
        m = re.search(r"array\(\[(.*?)\],\s*dtype=", s, flags=re.S)
        if m:
            inner = m.group(1)
            return [x.strip().strip("'\"") for x in re.findall(r"'([^']*)'|\"([^\"]*)\"", inner)
                    for x in [x[0] or x[1]] if x.strip().strip("'\"")]
        d = ast.literal_eval(s)
        return _parse_squad_answers_cell(d)
    except Exception:
        return []


def load_squad2(path: str, subsample: int, seed: int) -> list[dict]:
    path = Path(path)
    records: list[dict] = []

    # Prefer HuggingFace (clean schema); fall back to local zip CSV
    try:
        from datasets import load_dataset
        ds = load_dataset("rajpurkar/squad_v2", split="validation")
        _log(f"SQuAD2 via datasets: n={len(ds)}")
        rows = list(ds)
    except Exception as e:
        _log(f"datasets load failed ({e}); using local zip CSV")
        rows = []
        with zipfile.ZipFile(path) as z:
            df = pd.read_csv(z.open("validation.csv"))
        for _, r in df.iterrows():
            rows.append({
                "id": r["id"],
                "question": r["question"],
                "context": r["context"],
                "answers": {"text": _parse_squad_answers_cell(r["answers"])},
            })

    rng = np.random.default_rng(seed)
    if subsample and len(rows) > subsample:
        idx = rng.choice(len(rows), size=subsample, replace=False)
        rows = [rows[i] for i in sorted(idx.tolist())]

    for ex in rows:
        if hasattr(ex, "keys"):
            eid = ex["id"]
            q = ex["question"]
            ctx = ex["context"]
            ans_obj = ex["answers"]
        else:
            eid, q, ctx, ans_obj = ex["id"], ex["question"], ex["context"], ex["answers"]
        golds = _parse_squad_answers_cell(ans_obj) if not isinstance(ans_obj, dict) else [
            str(t).strip() for t in (ans_obj.get("text") or []) if str(t).strip()
        ]
        if hasattr(ans_obj, "get") and hasattr(ans_obj.get("text", None), "tolist"):
            golds = [str(t).strip() for t in ans_obj["text"].tolist() if str(t).strip()]
        is_unans = len(golds) == 0
        # HF squad_v2 also has answerable flag via empty answers
        records.append({
            "id": str(eid),
            "question": str(q).strip(),
            "context": str(ctx).strip() if ctx is not None else None,
            "gold_answers": golds,
            "is_unanswerable": bool(is_unans),
            "qtype": "squad2",
        })
    _log(f"SQuAD2 loaded: n={len(records)} unanswerable={sum(r['is_unanswerable'] for r in records)}")
    return records


def load_bioasq_factoid(path: str) -> list[dict]:
    path = Path(path)
    with zipfile.ZipFile(path) as z:
        # training13b.json lives under BioASQ-training13b/
        name = next(n for n in z.namelist() if n.endswith("training13b.json"))
        data = json.load(z.open(name))
    records = []
    for q in data["questions"]:
        if q.get("type") != "factoid":
            continue
        golds = _flatten_answers(q.get("exact_answer"))
        records.append({
            "id": str(q.get("id", q.get("body", "")[:40])),
            "question": str(q["body"]).strip(),
            "context": None,  # Task B factoid: question-only (docs are URLs)
            "gold_answers": golds,
            "is_unanswerable": False,
            "qtype": "factoid",
        })
    _log(f"BioASQ factoid loaded: n={len(records)}")
    return records


squad_records = load_squad2(CFG["squad_path"], CFG["squad_subsample"], CFG["seed"])
bioasq_records = load_bioasq_factoid(CFG["bioasq_path"])
DATASETS = {
    "squad2": squad_records,
    "bioasq": bioasq_records,
}

# Optional smoke truncation (also limits perturbation work)
import os as _os
if _os.environ.get("QA_SMOKE", "").strip() in {"1", "true", "True"}:
    _smoke_n = int(_os.environ.get("QA_SMOKE_N", "8"))
    _log(f"QA_SMOKE: truncating each dataset to {_smoke_n} instances")
    DATASETS = {k: v[:_smoke_n] for k, v in DATASETS.items()}
    squad_records = DATASETS["squad2"]
    bioasq_records = DATASETS["bioasq"]

print({k: len(v) for k, v in DATASETS.items()})
print("sample SQuAD:", {k: squad_records[0][k] for k in squad_records[0]})
print("sample BioASQ:", {k: bioasq_records[0][k] for k in bioasq_records[0]})


[2026-08-04 12:16:06 UTC] SQuAD2 via datasets: n=11873


[2026-08-04 12:16:06 UTC] SQuAD2 loaded: n=1000 unanswerable=505


[2026-08-04 12:16:06 UTC] BioASQ factoid loaded: n=1600


{'squad2': 1000, 'bioasq': 1600}
sample SQuAD: {'id': '56dddf4066d3e219004dad5f', 'question': 'Who was the duke in the battle of Hastings?', 'context': 'The Norman dynasty had a major political, cultural and military impact on medieval Europe and even the Near East. The Normans were famed for their martial spirit and eventually for their Christian piety, becoming exponents of the Catholic orthodoxy into which they assimilated. They adopted the Gallo-Romance language of the Frankish land they settled, their dialect becoming known as Norman, Normaund or Norman French, an important literary language. The Duchy of Normandy, which they formed by treaty with the French crown, was a great fief of medieval France, and under Richard I of Normandy was forged into a cohesive and formidable principality in feudal tenure. The Normans are noted both for their culture, such as their unique Romanesque architecture and musical traditions, and for their significant military accomplishments and innovatio

In [4]:
# =============================================================================
# 3–4) Prompt + generate_answer (greedy, logprobs, chat templates)
# =============================================================================

_LLAMA3_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
    "{{ content }}{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}{% endif %}"
)


def _ensure_chat_template(tokenizer) -> None:
    if getattr(tokenizer, "chat_template", None):
        return
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    unk = getattr(tokenizer, "unk_token_id", None)
    if eot is not None and eot != unk:
        tokenizer.chat_template = _LLAMA3_CHAT_TEMPLATE
        return
    tokenizer.chat_template = (
        "{{ bos_token }}{% for message in messages %}"
        "{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}{{ message['content'] }}"
        "{% endif %}{% endfor %}"
    )


def _arch_for(name: str) -> str:
    for s in MODEL_SPECS:
        if name in (s["key"], s["name"]):
            return s["arch"]
    return "causal"


def build_prompt(rec: dict, tok, name: str) -> str:
    """context+question if context else question; chat template for causal, raw for seq2seq."""
    if rec.get("context"):
        user = (
            "Answer the question using only the context. "
            "Reply with a short answer span only. "
            "If the question is unanswerable from the context, reply with 'unanswerable'.\n\n"
            f"Context: {rec['context']}\n\n"
            f"Question: {rec['question']}\n\n"
            "Answer:"
        )
    else:
        user = (
            "Answer the biomedical factoid question with a short exact answer only. "
            "Do not explain.\n\n"
            f"Question: {rec['question']}\n\n"
            "Answer:"
        )
    if _arch_for(name) == "seq2seq":
        return user
    _ensure_chat_template(tok)
    return tok.apply_chat_template(
        [{"role": "user", "content": user}],
        add_generation_prompt=True,
        tokenize=False,
    )


def _clean_answer(text: str) -> str:
    text = text.strip()
    for marker in ("[/INST]", "</s>", "<s>", "<|eot_id|>"):
        text = text.replace(marker, " ")
    if "Answer:" in text:
        text = text.split("Answer:")[-1]
    text = " ".join(text.split()).strip(" \"'`")
    return text[:300]


def generate_answer(model, tok, prompt: str, name: str):
    """Greedy decode -> (text, mean_token_logprob). Causal slices off prompt tokens."""
    arch = _arch_for(name)
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
    try:
        first_dev = next(model.parameters()).device
    except StopIteration:
        first_dev = torch.device("cuda:0")
    enc = {k: v.to(first_dev) for k, v in enc.items()}
    input_len = enc["input_ids"].shape[-1]

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=CFG["max_new_tokens"],
            do_sample=False,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tok.eos_token_id if tok.eos_token_id is not None else tok.pad_token_id,
        )

    seq = out.sequences[0]
    if arch == "causal":
        gen_ids = seq[input_len:]
    else:
        gen_ids = seq
        if tok.pad_token_id is not None:
            gen_ids = gen_ids[gen_ids != tok.pad_token_id]
        if len(gen_ids) and tok.bos_token_id is not None and int(gen_ids[0]) == tok.bos_token_id:
            gen_ids = gen_ids[1:]

    gen_ids = gen_ids.detach().cpu()
    text = _clean_answer(tok.decode(gen_ids, skip_special_tokens=True))

    scores = out.scores
    if not scores or len(gen_ids) == 0:
        return text, float("nan")
    logps = []
    n_steps = min(len(scores), int(gen_ids.shape[0]))
    for t in range(n_steps):
        logits = scores[t][0]
        lp = torch.log_softmax(logits.float(), dim=-1)
        tid = int(gen_ids[t].item())
        logps.append(float(lp[tid].item()))
    mean_lp = float(np.mean(logps)) if logps else float("nan")
    return text, mean_lp


print("build_prompt / generate_answer ready")


build_prompt / generate_answer ready


In [5]:
# =============================================================================
# 5) Question perturbations: G1–G5 ON, G6 OFF (UMLS synonym gate undefined)
# =============================================================================
from sentence_transformers import SentenceTransformer

_log("Loading perturbation generators + gates (G1–G5; G6 off) ...")

_mt_en_de_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de", local_files_only=True)
_mt_en_de = MarianMTModel.from_pretrained(
    "Helsinki-NLP/opus-mt-en-de", weights_only=False, local_files_only=True
).to(DEVICE).eval()
_mt_de_en_tok = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en", local_files_only=True)
_mt_de_en = MarianMTModel.from_pretrained(
    "Helsinki-NLP/opus-mt-de-en", weights_only=False, local_files_only=True
).to(DEVICE).eval()

_para_tok = T5Tokenizer.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base", local_files_only=True)
_para_model = T5ForConditionalGeneration.from_pretrained(
    "humarin/chatgpt_paraphraser_on_T5_base", weights_only=False, local_files_only=True
).to(DEVICE).eval()

# G2 / clustering NLI — resolve entailment index from id2label (NOT hardcoded)
from transformers import AutoModelForSequenceClassification, AutoTokenizer as _NLITok
_nli_tok = _NLITok.from_pretrained(CFG["nli_model"])
_nli_model = AutoModelForSequenceClassification.from_pretrained(CFG["nli_model"]).to(DEVICE).eval()
print("NLI id2label:", dict(_nli_model.config.id2label))
_ENTAIL_IDX = next(
    int(i) for i, lab in _nli_model.config.id2label.items()
    if "entail" in str(lab).lower()
)
_CONTRA_IDX = next(
    (int(i) for i, lab in _nli_model.config.id2label.items()
     if "contrad" in str(lab).lower()),
    None,
)
print(f"Resolved ENTAIL_IDX={_ENTAIL_IDX}  CONTRA_IDX={_CONTRA_IDX}")
# MiniLM expected order: contradiction=0, entailment=1, neutral=2
if "MiniLM" in CFG["nli_model"] or "nli-MiniLM" in CFG["nli_model"]:
    assert _ENTAIL_IDX == 1, (
        f"Silent killer: MiniLM entailment index is {_ENTAIL_IDX}, expected 1 "
        f"(do NOT use roberta-large-mnli index 2). id2label={dict(_nli_model.config.id2label)}"
    )

_nli_pipe = pipeline(
    "text-classification",
    model=_nli_model,
    tokenizer=_nli_tok,
    device=0,
    top_k=None,  # return all labels
)

_sbert = SentenceTransformer("all-MiniLM-L6-v2")  # cached; hub offline via env

# G4 LanguageTool (fallback heuristic if Java/LT fails)
_lt_tool = None
try:
    import language_tool_python
    _lt_tool = language_tool_python.LanguageTool("en-US")
    _log("LanguageTool loaded for G4")
except Exception as _lt_err:
    _log(f"LanguageTool unavailable — G4 heuristic fallback: {_lt_err}")


def levenshtein_norm(a: str, b: str) -> float:
    la, lb = len(a), len(b)
    if la == 0 and lb == 0:
        return 0.0
    dp = list(range(lb + 1))
    for i in range(1, la + 1):
        prev, dp[0] = dp[:], i
        for j in range(1, lb + 1):
            dp[j] = prev[j - 1] if a[i - 1] == b[j - 1] else 1 + min(prev[j], dp[j - 1], prev[j - 1])
    return dp[lb] / max(la, lb)


def back_translate(text: str) -> str:
    try:
        enc = _mt_en_de_tok([text], return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            de_ids = _mt_en_de.generate(**enc, max_new_tokens=128)
        de = _mt_en_de_tok.decode(de_ids[0], skip_special_tokens=True)
        enc2 = _mt_de_en_tok([de], return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            en_ids = _mt_de_en.generate(**enc2, max_new_tokens=128)
        return _mt_de_en_tok.decode(en_ids[0], skip_special_tokens=True)
    except Exception:
        return text


def paraphrase(text: str) -> str:
    try:
        enc = _para_tok(
            f"paraphrase: {text}", return_tensors="pt", truncation=True, max_length=256
        ).to(DEVICE)
        with torch.no_grad():
            out = _para_model.generate(**enc, max_new_tokens=128, num_beams=4, do_sample=False)
        return _para_tok.decode(out[0], skip_special_tokens=True)
    except Exception:
        return text


def synonym_sub(text: str) -> str:
    try:
        import nltk
        from nltk.corpus import wordnet
        nltk.download("wordnet", quiet=True)
        words = text.split()
        new_words, changed = [], 0
        for w in words:
            syns = wordnet.synsets(w.lower())
            if syns and changed < 3:
                lemmas = [
                    l.name().replace("_", " ")
                    for l in syns[0].lemmas()
                    if l.name().lower() != w.lower()
                ]
                if lemmas:
                    new_words.append(lemmas[0])
                    changed += 1
                    continue
            new_words.append(w)
        return " ".join(new_words)
    except Exception:
        return text


def syntactic_reorder(text: str) -> str:
    try:
        import spacy
        spacy.require_cpu()
        nlp = spacy.load("en_core_web_sm")
        doc = nlp(text)
        sents = list(doc.sents)
        if len(sents) > 1:
            return " ".join(str(s) for s in reversed(sents))
        tokens = [t.text for t in doc]
        if len(tokens) > 6:
            mid = len(tokens) // 2
            return " ".join(tokens[mid:] + tokens[:mid])
        return text
    except Exception:
        return text


def gate_g1(orig: str, pert: str) -> bool:
    try:
        embs = _sbert.encode([orig, pert], normalize_embeddings=True)
        return float(embs[0] @ embs[1]) >= 0.85
    except Exception:
        return True


def gate_g2(orig: str, pert: str) -> bool:
    """Reject high-confidence contradictions (same G2 as concept grid)."""
    try:
        res = _nli_pipe(f"{orig} [SEP] {pert}", truncation=True, max_length=256)
        # top_k=None -> list[list[{label,score}]]
        scores = res[0] if isinstance(res[0], list) else res
        by = {d["label"].upper(): float(d["score"]) for d in scores}
        return not (by.get("CONTRADICTION", 0.0) >= CFG["nli_threshold"])
    except Exception:
        return True


def gate_g3(orig: str, pert: str) -> bool:
    neg_words = {"not", "no", "never", "none", "neither", "nor", "without", "cannot", "can't", "won't", "don't"}
    o = {w for w in orig.lower().split() if w in neg_words}
    p = {w for w in pert.lower().split() if w in neg_words}
    return o == p


def grammar_score(text: str) -> float:
    toks = max(len(text.split()), 1)
    if _lt_tool is not None:
        try:
            matches = _lt_tool.check(text)
            # 1 - errors/token, clipped
            return float(max(0.0, 1.0 - len(matches) / toks))
        except Exception:
            pass
    # heuristic: fraction of alphabetic tokens
    words = text.split()
    if not words:
        return 0.0
    ok = sum(1 for w in words if any(c.isalpha() for c in w))
    return ok / len(words)


def gate_g4(pert: str) -> bool:
    return grammar_score(pert) >= 0.50


def gate_g5(mag: float) -> bool:
    return 0.05 <= mag <= 0.60


def accepted_question_perturbations(rec: dict) -> list[str]:
    """Generate candidates; keep those that pass G1–G5. G6 stays off."""
    text = rec["question"]
    methods = [
        back_translate, back_translate,
        paraphrase, paraphrase,
        synonym_sub, synonym_sub,
        syntactic_reorder, syntactic_reorder,
    ]
    accepted, seen = [], set()
    for fn in methods[: CFG["k_perturb_attempts"]]:
        pert = fn(text).strip()
        if not pert or pert == text:
            continue
        key = pert.lower()
        if key in seen:
            continue
        mag = levenshtein_norm(text, pert)
        if not gate_g5(mag):
            continue
        if not gate_g3(text, pert):
            continue
        if not gate_g4(pert):
            continue
        if not gate_g1(text, pert):
            continue
        if not gate_g2(text, pert):
            continue
        # G6 OFF — intentionally no UMLS / answer-preservation gate
        seen.add(key)
        accepted.append(pert)
    return accepted


_log("Gates ready (G1–G5; G6 off)")


[2026-08-04 12:16:07 UTC] Loading perturbation generators + gates (G1–G5; G6 off) ...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 43011.66it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 44057.10it/s]

...

NLI id2label: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Resolved ENTAIL_IDX=1  CONTRA_IDX=0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5555.87it/s]

[2026-08-04 12:16:20 UTC] LanguageTool unavailable — G4 heuristic fallback: No java install detected. Please install java to use language-tool-python.


[2026-08-04 12:16:20 UTC] Gates ready (G1–G5; G6 off)


In [6]:
# =============================================================================
# Cache accepted question perturbations (once per dataset)
# =============================================================================

def pert_cache_path(dataset: str) -> Path:
    return INTER_DIR / f"qa_question_perturbations_{dataset}.csv"


def build_or_load_perturbations(dataset: str, records: list[dict]) -> dict[str, list[str]]:
    path = pert_cache_path(dataset)
    meta = path.with_suffix(".meta.json")
    needed = {str(r["id"]) for r in records}

    if path.exists() and meta.exists():
        import json as _json
        meta_obj = _json.loads(meta.read_text())
        cached_ids = set(meta_obj.get("ids", []))
        if needed <= cached_ids:
            df = pd.read_csv(path) if path.stat().st_size > 0 else pd.DataFrame()
            out = defaultdict(list)
            if len(df):
                for _, r in df.iterrows():
                    out[str(r["id"])].append(str(r["pert_question"]))
            for i in needed:
                out.setdefault(i, [])
            _log(f"Loaded pert cache {path.name}: {len(df)} rows / {len(needed)} ids")
            return {i: out[i] for i in needed}
        _log(f"Pert cache {path.name} incomplete for this run — regenerating")

    rows = []
    for i, rec in enumerate(tqdm(records, desc=f"perts:{dataset}")):
        accepted = accepted_question_perturbations(rec)
        for j, p in enumerate(accepted):
            rows.append({
                "id": rec["id"],
                "dataset": dataset,
                "pert_idx": j,
                "pert_question": p,
                "m": len(accepted),
            })
        if (i + 1) % 50 == 0:
            _log(f"  {dataset} [{i+1}/{len(records)}]")
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    import json as _json
    meta.write_text(_json.dumps({"ids": [str(r["id"]) for r in records], "n": len(records)}))
    out = defaultdict(list)
    for _, r in df.iterrows():
        out[str(r["id"])].append(str(r["pert_question"]))
    for rec in records:
        out.setdefault(str(rec["id"]), [])
    m_list = [len(out[str(r["id"])]) for r in records]
    n_excl = sum(1 for m in m_list if m < CFG["min_m"])
    _log(f"Saved {path}: accepted_rows={len(df)}  excluded_m<min_m={n_excl}/{len(records)}")
    return dict(out)


# Build caches (skip if already present and complete)
PERTS = {}
for ds_name, recs in DATASETS.items():
    PERTS[ds_name] = build_or_load_perturbations(ds_name, recs)

# Free heavy generators (keep NLI for clustering)
del _mt_en_de, _mt_de_en, _para_model, _sbert
if _lt_tool is not None:
    try:
        _lt_tool.close()
    except Exception:
        pass
_lt_tool = None
gc.collect()
torch.cuda.empty_cache()
_log("Perturbation generators freed; NLI retained for answer clustering")


perts:squad2:   0%|          | 0/1000 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[2026-08-04 13:09:46 UTC]   bioasq [1600/1600]


perts:bioasq: 100%|██████████| 1600/1600 [33:44<00:00,  1.39s/it]

perts:bioasq: 100%|██████████| 1600/1600 [33:44<00:00,  1.27s/it]

[2026-08-04 13:09:46 UTC] Saved /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa/intermediate/qa_question_perturbations_bioasq.csv: accepted_rows=2964  excluded_m<min_m=1208/1600


[2026-08-04 13:09:47 UTC] Perturbation generators freed; NLI retained for answer clustering


In [7]:
# =============================================================================
# 6–8) Answer clustering (bidirectional NLI), entropy, correctness
# =============================================================================

class UnionFind:
    def __init__(self, n: int):
        self.p = list(range(n))
        self.r = [0] * n

    def find(self, x: int) -> int:
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.r[ra] < self.r[rb]:
            self.p[ra] = rb
        elif self.r[ra] > self.r[rb]:
            self.p[rb] = ra
        else:
            self.p[rb] = ra
            self.r[ra] += 1


def entailment_prob(a: str, b: str) -> float:
    """P(entailment | a entails b) using softmax at resolved ENTAIL_IDX from id2label."""
    enc = _nli_tok(
        a, b, return_tensors="pt", truncation=True, max_length=256, padding=True
    )
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = _nli_model(**enc).logits[0].float()
        probs = torch.softmax(logits, dim=-1)
    return float(probs[_ENTAIL_IDX].item())


def cluster_answers(answers: list[str], nli=None, thr: float | None = None) -> list[int]:
    """Union-find: string-match OR bidirectional entailment >= thr."""
    thr = CFG["nli_threshold"] if thr is None else thr
    n = len(answers)
    uf = UnionFind(n)
    lows = [a.strip().lower() for a in answers]
    for i in range(n):
        for j in range(i + 1, n):
            if lows[i] == lows[j]:
                uf.union(i, j)
                continue
            if not lows[i] or not lows[j]:
                # both empty already matched; one empty != other
                continue
            e_ij = entailment_prob(answers[i], answers[j])
            e_ji = entailment_prob(answers[j], answers[i])
            if e_ij >= thr and e_ji >= thr:
                uf.union(i, j)
    roots = [uf.find(i) for i in range(n)]
    remap, labels, k = {}, [], 0
    for r in roots:
        if r not in remap:
            remap[r] = k
            k += 1
        labels.append(remap[r])
    return labels


def normalised_entropy(labels: list[int], m: int) -> tuple[float, float, int]:
    """H / log2(m+1) using THIS instance's accepted m (not log2(9))."""
    total = len(labels)
    assert total == m + 1, f"expected m+1={m+1} labels, got {total}"
    counts = Counter(labels)
    n_clusters = len(counts)
    p = np.array([c / total for c in counts.values()], dtype=float)
    H = float(-(p * np.log2(p + 1e-15)).sum())
    denom = math.log2(m + 1)
    H_norm = float(H / denom) if denom > 0 else 0.0
    return H, H_norm, n_clusters


# ---- SQuAD official-style normalise / EM / F1 ----
_ARTICLES = re.compile(r"\b(a|an|the)\b", re.I)
_PUNCT = set(string.punctuation)


def squad_normalize(s: str) -> str:
    s = s.lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = _ARTICLES.sub(" ", s)
    s = " ".join(s.split())
    return s


def _is_empty_pred(s: str) -> bool:
    n = squad_normalize(s)
    if not n:
        return True
    return n in {"unanswerable", "no answer", "none", "n/a", "unknown", "cannot answer"}


def token_f1(pred: str, gold: str) -> float:
    pt, gt = squad_normalize(pred).split(), squad_normalize(gold).split()
    if not pt and not gt:
        return 1.0
    if not pt or not gt:
        return 0.0
    common = Counter(pt) & Counter(gt)
    num = sum(common.values())
    if num == 0:
        return 0.0
    precision = num / len(pt)
    recall = num / len(gt)
    return 2 * precision * recall / (precision + recall)


def squad_correctness(pred: str, golds: list[str], is_unanswerable: bool) -> tuple[bool, float, float]:
    if is_unanswerable:
        ok = _is_empty_pred(pred)
        return ok, float(ok), float(ok)
    if _is_empty_pred(pred) and golds:
        return False, 0.0, 0.0
    em = max(float(squad_normalize(pred) == squad_normalize(g)) for g in golds) if golds else 0.0
    f1 = max(token_f1(pred, g) for g in golds) if golds else 0.0
    return bool(em == 1.0), em, f1


def bioasq_correctness(pred: str, golds: list[str]) -> tuple[bool, float, float]:
    pn = squad_normalize(pred)
    ok = False
    for g in golds:
        gn = squad_normalize(g)
        if gn == pn or (gn and gn in pn):
            ok = True
            break
    return ok, float(ok), float(ok)


print("cluster_answers / entropy / correctness ready")


cluster_answers / entropy / correctness ready


In [8]:
# Optional smoke: QA_SMOKE=1 limits instances for a short pipeline check
import os as _os
if _os.environ.get("QA_SMOKE", "").strip() in {"1", "true", "True"}:
    _smoke_n = int(_os.environ.get("QA_SMOKE_N", "8"))
    _log(f"QA_SMOKE: truncating each dataset to {_smoke_n} instances")
    for _ds in list(DATASETS.keys()):
        DATASETS[_ds] = DATASETS[_ds][:_smoke_n]
        # rebuild pert map subset only (full cache still used)
        PERTS[_ds] = {r["id"]: PERTS[_ds].get(r["id"], []) for r in DATASETS[_ds]}

# =============================================================================
# 9) Main loop: per (dataset, model, instance)
# =============================================================================
RESULT_COLS = [
    "id", "model", "dataset", "included", "m", "n_clusters", "entropy", "norm_entropy",
    "is_zero", "pred", "correct", "em", "f1", "confidence", "is_unanswerable", "qtype",
]


def result_path(dataset: str, model_key: str) -> Path:
    # Isolate smoke result CSVs from full-run outputs
    base = INTER_DIR if _os.environ.get("QA_SMOKE", "").strip() in {"1", "true", "True"} else OUT_DIR
    return base / f"qa_results_{dataset}_{model_key}.csv"


def run_one_model(dataset: str, records: list[dict], pert_map: dict, spec: dict) -> pd.DataFrame:
    out_path = result_path(dataset, spec["key"])
    if out_path.exists() and out_path.stat().st_size > 0:
        df_exist = pd.read_csv(out_path)
        _log(f"SKIP {spec['name']} / {dataset} — exists {out_path.name} ({len(df_exist)} rows)")
        return df_exist

    _log(f"LOAD {spec['name']} for {dataset}")
    model, tok = load_generative(spec["key"])
    rows = []
    n_excl = 0

    for rec in tqdm(records, desc=f"{dataset}:{spec['key']}"):
        accepted = list(pert_map.get(rec["id"], []))
        m = len(accepted)
        if m < CFG["min_m"]:
            n_excl += 1
            rows.append({
                "id": rec["id"], "model": spec["name"], "dataset": dataset,
                "included": False, "m": m, "n_clusters": np.nan,
                "entropy": np.nan, "norm_entropy": np.nan, "is_zero": np.nan,
                "pred": "", "correct": np.nan, "em": np.nan, "f1": np.nan,
                "confidence": np.nan,
                "is_unanswerable": rec["is_unanswerable"], "qtype": rec["qtype"],
            })
            continue

        variants_q = [rec["question"]] + accepted  # original first
        answers, confs = [], []
        for qi, q in enumerate(variants_q):
            rec_var = dict(rec)
            rec_var["question"] = q
            prompt = build_prompt(rec_var, tok, spec["key"])
            text, mean_lp = generate_answer(model, tok, prompt, spec["key"])
            answers.append(text)
            confs.append(mean_lp)

        labels = cluster_answers(answers, _nli_pipe, CFG["nli_threshold"])
        H, Hn, n_cl = normalised_entropy(labels, m)
        pred0, conf0 = answers[0], confs[0]

        if dataset == "squad2":
            correct, em, f1 = squad_correctness(pred0, rec["gold_answers"], rec["is_unanswerable"])
        else:
            correct, em, f1 = bioasq_correctness(pred0, rec["gold_answers"])

        rows.append({
            "id": rec["id"], "model": spec["name"], "dataset": dataset,
            "included": True, "m": m, "n_clusters": n_cl,
            "entropy": H, "norm_entropy": Hn, "is_zero": bool(Hn == 0.0 or n_cl == 1),
            "pred": pred0, "correct": bool(correct), "em": em, "f1": f1,
            "confidence": conf0,
            "is_unanswerable": rec["is_unanswerable"], "qtype": rec["qtype"],
        })

    free_model(model)
    df = pd.DataFrame(rows)[RESULT_COLS]
    df.to_csv(out_path, index=False)
    n_inc = int(df["included"].sum())
    _log(
        f"DONE {spec['name']} / {dataset}: included={n_inc} excluded={n_excl} "
        f"-> {out_path.name}"
    )
    return df


all_frames = []
exclusion_report = []

for ds_name, recs in DATASETS.items():
    # report exclusions once (model-independent: based on m)
    m_map = {rid: len(PERTS[ds_name].get(rid, [])) for rid in [r["id"] for r in recs]}
    # also keys from records
    m_list = [len(PERTS[ds_name].get(r["id"], [])) for r in recs]
    n_excl = sum(1 for m in m_list if m < CFG["min_m"])
    exclusion_report.append({
        "dataset": ds_name,
        "n_instances": len(recs),
        "n_excluded_m_lt_min": n_excl,
        "n_included": len(recs) - n_excl,
        "min_m": CFG["min_m"],
        "mean_m": float(np.mean(m_list)) if m_list else 0.0,
    })
    _log(f"EXCLUSIONS {ds_name}: excluded={n_excl}/{len(recs)} (m < {CFG['min_m']})")

    for spec in MODEL_SPECS:
        df_m = run_one_model(ds_name, recs, PERTS[ds_name], spec)
        all_frames.append(df_m)

pd.DataFrame(exclusion_report).to_csv(INTER_DIR / "qa_exclusion_report.csv", index=False) if _os.environ.get("QA_SMOKE", "").strip() in {"1", "true", "True"} else pd.DataFrame(exclusion_report).to_csv(OUT_DIR / "qa_exclusion_report.csv", index=False)
combined = pd.concat(all_frames, ignore_index=True)
combined.to_csv(OUT_DIR / "qa_results_combined.csv", index=False)
_log(f"Combined results: {len(combined)} rows -> qa_results_combined.csv")
print(combined.groupby(["dataset", "model"])["included"].agg(["sum", "count"]))


[2026-08-04 13:09:47 UTC] EXCLUSIONS squad2: excluded=861/1000 (m < 3)


[2026-08-04 13:09:47 UTC] LOAD BioMistral-7B for squad2


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/291 [00:00<00:57,  5.06it/s]

...

bioasq:flan-t5-base: 100%|██████████| 1600/1600 [02:33<00:00, 13.20it/s]

bioasq:flan-t5-base: 100%|██████████| 1600/1600 [02:33<00:00, 10.40it/s]

[2026-08-04 14:22:53 UTC] DONE FLAN-T5-base / bioasq: included=392 excluded=1208 -> qa_results_bioasq_flan-t5-base.csv


[2026-08-04 14:22:53 UTC] Combined results: 13000 rows -> qa_results_combined.csv


                                  sum  count
dataset model                               
bioasq  BioMistral-7B             392   1600
        FLAN-T5-base              392   1600
        Meta-Llama-3-8B-Instruct  392   1600
        Mistral-7B-Instruct-v0.1  392   1600
        OpenBioLLM-8B             392   1600
squad2  BioMistral-7B             139   1000
        FLAN-T5-base              139   1000
        Meta-Llama-3-8B-Instruct  139   1000
        Mistral-7B-Instruct-v0.1  139   1000
        OpenBioLLM-8B             139   1000


## 10) Analysis (per model × dataset)

- `zero_inflation = mean(is_zero)` — contrast vs CUI-entropy zeros (~50–75%; OpenBioLLM ~0.98 on CADEC)
- AURC for entropy / confidence / random
- `entropy_beats_conf` reported honestly (not assumed universal)
- SQuAD only: `unans_auroc_entropy = roc_auc_score(is_unanswerable, norm_entropy)`
- Risk–coverage PNGs ordered by ascending `norm_entropy`


In [9]:
# =============================================================================
# 10) Analysis
# =============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

combined = pd.read_csv(OUT_DIR / "qa_results_combined.csv")
inc = combined[combined["included"] == True].copy()
_log(f"Analysis on included rows: {len(inc)}")


def aurc(score: np.ndarray, correct: np.ndarray) -> float:
    """Sort ascending (most-confident / least-risky first); mean prefix risk = 1 - acc."""
    score = np.asarray(score, dtype=float)
    correct = np.asarray(correct, dtype=float)
    order = np.argsort(score)  # low score kept first
    c = correct[order]
    risks = []
    for k in range(1, len(c) + 1):
        risks.append(1.0 - float(c[:k].mean()))
    return float(np.mean(risks)) if risks else float("nan")


def risk_coverage_curve(score: np.ndarray, correct: np.ndarray):
    score = np.asarray(score, dtype=float)
    correct = np.asarray(correct, dtype=float)
    order = np.argsort(score)
    c = correct[order]
    cov, risk = [], []
    n = len(c)
    for k in range(1, n + 1):
        cov.append(k / n)
        risk.append(1.0 - float(c[:k].mean()))
    return np.array(cov), np.array(risk)


summary_rows = []
rng = np.random.default_rng(CFG["seed"])

for (dataset, model), g in inc.groupby(["dataset", "model"], sort=True):
    g = g.dropna(subset=["norm_entropy", "correct"])
    if len(g) < 10:
        _log(f"skip {dataset}/{model}: n={len(g)}")
        continue
    correct = g["correct"].astype(float).values
    ent = g["norm_entropy"].astype(float).values
    # confidence: mean logprob (higher/closer to 0 = better) -> negate so high = risky
    conf = g["confidence"].astype(float).values
    conf_risk = -conf
    # random baseline AURC equals overall error rate
    overall_err = 1.0 - float(correct.mean())
    a_ent = aurc(ent, correct)
    a_conf = aurc(conf_risk, correct)
    a_rand = overall_err  # E[risk] under random order ≈ 1 - accuracy

    row = {
        "dataset": dataset,
        "model": model,
        "n_included": len(g),
        "zero_inflation": float(g["is_zero"].astype(float).mean()),
        "mean_norm_entropy": float(ent.mean()),
        "mean_confidence": float(np.nanmean(conf)),
        "accuracy": float(correct.mean()),
        "aurc_entropy": a_ent,
        "aurc_confidence": a_conf,
        "aurc_random": a_rand,
        "entropy_beats_conf": bool(a_ent < a_conf),
        "unans_auroc_entropy": np.nan,
    }

    if dataset == "squad2":
        y = g["is_unanswerable"].astype(int).values
        if y.min() != y.max():
            try:
                row["unans_auroc_entropy"] = float(roc_auc_score(y, ent))
            except Exception as e:
                _log(f"AUROC failed {model}: {e}")
                row["unans_auroc_entropy"] = np.nan
        else:
            row["unans_auroc_entropy"] = np.nan

    summary_rows.append(row)
    _log(
        f"{dataset:7s} {model:28s}  zero={row['zero_inflation']:.3f}  "
        f"AURC_ent={a_ent:.4f} AURC_conf={a_conf:.4f}  "
        f"beats_conf={row['entropy_beats_conf']}  "
        f"unans_auroc={row['unans_auroc_entropy']}"
    )

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "qa_summary.csv", index=False)
_log(f"Wrote {OUT_DIR / 'qa_summary.csv'}")
print(summary.to_string(index=False))

# Honest note: do NOT claim entropy beats confidence universally
n_beat = int(summary["entropy_beats_conf"].sum())
_log(
    f"entropy_beats_conf in {n_beat}/{len(summary)} model×dataset cells "
    f"(not universal — same posture as the concept grid)"
)

# Risk–coverage PNGs (entropy ordering)
for dataset in ["squad2", "bioasq"]:
    sub = inc[inc["dataset"] == dataset]
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    for model, g in sub.groupby("model"):
        g = g.dropna(subset=["norm_entropy", "correct"])
        if len(g) < 10:
            continue
        cov, risk = risk_coverage_curve(
            g["norm_entropy"].astype(float).values,
            g["correct"].astype(float).values,
        )
        ax.plot(cov, risk, label=model, linewidth=1.8)
    ax.set_xlabel("Coverage (fraction retained, low→high entropy)")
    ax.set_ylabel("Risk (1 − accuracy)")
    ax.set_title(f"QA answer-level entropy risk–coverage ({dataset})")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig_path = OUT_DIR / f"qa_risk_coverage_{dataset}.png"
    fig.savefig(fig_path, dpi=150)
    plt.close(fig)
    _log(f"Wrote {fig_path}")

print("Analysis complete.")


[2026-08-04 14:22:54 UTC] Analysis on included rows: 2655


[2026-08-04 14:22:54 UTC] bioasq  BioMistral-7B                 zero=0.097  AURC_ent=0.6373 AURC_conf=0.6708  beats_conf=True  unans_auroc=nan


[2026-08-04 14:22:54 UTC] bioasq  FLAN-T5-base                  zero=0.151  AURC_ent=0.9946 AURC_conf=0.9869  beats_conf=False  unans_auroc=nan


[2026-08-04 14:22:54 UTC] bioasq  Meta-Llama-3-8B-Instruct      zero=0.283  AURC_ent=0.6621 AURC_conf=0.5868  beats_conf=False  unans_auroc=nan


[2026-08-04 14:22:54 UTC] bioasq  Mistral-7B-Instruct-v0.1      zero=0.087  AURC_ent=0.7361 AURC_conf=0.7539  beats_conf=True  unans_auroc=nan


...

[2026-08-04 14:22:54 UTC] Wrote /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa/qa_summary.csv


dataset                    model  n_included  zero_inflation  mean_norm_entropy  mean_confidence  accuracy  aurc_entropy  aurc_confidence  aurc_random  entropy_beats_conf  unans_auroc_entropy
 bioasq            BioMistral-7B         392        0.096939           0.680631        -0.312846  0.227041      0.637288         0.670824     0.772959                True                  NaN
 bioasq             FLAN-T5-base         392        0.150510           0.566340        -1.199269  0.007653      0.994584         0.986903     0.992347               False                  NaN
 bioasq Meta-Llama-3-8B-Instruct         392        0.283163           0.455210        -0.311074  0.239796      0.662091         0.586808     0.760204               False                  NaN
 bioasq Mistral-7B-Instruct-v0.1         392        0.086735           0.690398        -0.206316  0.178571      0.736137         0.753926     0.821429                True                  NaN
 bioasq            OpenBioLLM-8B        

[2026-08-04 14:22:54 UTC] Wrote /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa/qa_risk_coverage_squad2.png


[2026-08-04 14:22:54 UTC] Wrote /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/qa/qa_risk_coverage_bioasq.png


Analysis complete.
